# Module 5.4 — Hybrid Search

Combines:
- **Dense (semantic)** — vector similarity via embeddings
- **Sparse (keyword)** — BM25 exact/lexical matching

Hybrid beats pure semantic on: rare keywords, product codes, names, technical terms.

LangChain `EnsembleRetriever` merges results with a weighted `alpha` parameter.

In [ ]:
# !pip install rank_bm25
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

docs = [
    Document(page_content="GPT-4o is a multimodal model from OpenAI released in 2024."),
    Document(page_content="Claude 3.5 Sonnet is an AI assistant made by Anthropic."),
    Document(page_content="Gemini 1.5 Pro is Google's long-context multimodal model."),
    Document(page_content="LLaMA 3 is Meta's open-source large language model."),
    Document(page_content="Mistral 7B is a compact but powerful open-source LLM."),
]

# ── BM25 (sparse) retriever ────────────────────────────────────────────────────
bm25_retriever = BM25Retriever.from_documents(docs, k=3)

# ── Dense (semantic) retriever ────────────────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="hybrid_demo")
dense_retriever = vs.as_retriever(search_kwargs={"k": 3})

# ── Ensemble (hybrid) retriever ───────────────────────────────────────────────
ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]   # 40% BM25, 60% semantic
)

for query in ["GPT-4o multimodal capabilities", "open source LLM Meta"]:
    print(f"\nQuery: '{query}'")
    bm25_res    = bm25_retriever.invoke(query)
    dense_res   = dense_retriever.invoke(query)
    hybrid_res  = ensemble.invoke(query)
    print(f"  BM25   : {[d.page_content[:45] for d in bm25_res]}")
    print(f"  Dense  : {[d.page_content[:45] for d in dense_res]}")
    print(f"  Hybrid : {[d.page_content[:45] for d in hybrid_res]}")
